In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
import pandas as pd
import xgboost as xgb
import glob

# 1. LOAD DATA
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path = glob.glob('/kaggle/input/**/test.csv', recursive=True)[0]
sub_path = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)[0]

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

# 2. Identify Target and Map to Numbers
target = list(set(train.columns) - set(test.columns))[0]
# This turns 'Low', 'Medium', 'High' into 0, 1, 2
label_map = {val: i for i, val in enumerate(sorted(train[target].unique()))}
reverse_map = {i: val for val, i in label_map.items()}
train[target] = train[target].map(label_map)

# 3. Encoding Features
all_data = pd.concat([train.drop(target, axis=1), test], axis=0)
all_data_encoded = pd.get_dummies(all_data)

X = all_data_encoded.iloc[:len(train), :]
X_test = all_data_encoded.iloc[len(train):, :]
y = train[target]

if 'id' in X.columns:
    X = X.drop('id', axis=1)
    X_test = X_test.drop('id', axis=1)

# 4. XGBoost Model
model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6, tree_method='hist')

# 5. Train & Predict
print("Training...")
model.fit(X, y)
preds = model.predict(X_test)

# 6. Map Numbers back to Words and Save
submission[target] = [reverse_map[p] for p in preds]
submission.to_csv('submission.csv', index=False)
print("SUCCESS! File 'submission.csv' is ready.")

Training...
SUCCESS! File 'submission.csv' is ready.
